## SAT SOLVING EXERCISE - DA7067-Computational Mathematics

In [1]:
!pip install python-sat

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import random
def random_graph(number_of_vertices=10, prob_for_edge=50):
    """
    Generate a random graph with "number_of_vertices" vertices
    (default is 10 vertices) with probability for an edge between
    two vertices set to "prob_for_edge" (default is 50% of
    vertices will have edges between them).

    Parameters:
        number_of_vertices (int): A positive integer
        prob_for_edge (int): An integer from 0 to 100

    Returns:
        out (str): String containing a graph in the exercise format
    """

    out = ""
    number_of_edges = 0

    for v in range(1, number_of_vertices + 1):
        out += str(v)

        for w in range(v + 1, number_of_vertices + 1):
            r = random.randint(0, 100)

            if r < prob_for_edge:
                out += " " + str(w)
                number_of_edges += 1

        out += "\n"

    out = f"{number_of_vertices} {number_of_edges}\n" + out[:-1]

    return out

In [3]:
print(random_graph(5, 50))

5 6
1 2 3 4 5
2
3 4 5
4
5


In [4]:
print(random_graph( ))

10 25
1 4 5 8 9 10
2 4 5 6 8 10
3 6 9 10
4 5 7 8 10
5 10
6 9
7 8 9 10
8 9 10
9 10
10


## Problem 1 - Graph Coloring using SAT

In [5]:
from pysat.formula import CNF
from pysat.card import CardEnc
from pysat.solvers import Solver
import random
import time

## Problem 1a

In [6]:
#Read the graph
#satgraph as filename

def read_graph(satgraph):
    """
    Read the graph.

    Returns:
        n     - number of vertices
        edges - list of edges
    """

    with open(satgraph, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    n, m = map(int, lines[0].split())

    edges = []
    
    for line in lines[1:]:
        values = list(map(int, line.split()))

        v = values[0]

        for w in values[1:]:
            edges.append((v, w))

    return n, edges


In [7]:
#Create SAT variables/literals

def color_variable(vertex, color, k):
    """
    Give each (vertex, color) pair a unique positive integer.
    """

    return (vertex - 1) * k + color

In [8]:
print(color_variable(1, 1, 3))
print(color_variable(1, 2, 3))
print(color_variable(2, 1, 3))

1
2
4


In [9]:
#build the CNF formula
def graph_coloring_cnf(satgraph, k):
    """
    Convert graph k-coloring into a CNF formula.

    Returns:
        CNF formula
    """
    n, edges = read_graph(satgraph)
    cnf = CNF()
 # Each vertex must have at least one color
    for v in range(1, n + 1):
        clause = []
        for c in range(1, k + 1):
            clause.append(color_variable(v, c, k))
        cnf.append(clause)
    # Each vertex can have at most one color
    for v in range(1, n + 1):
        for c1 in range(1, k + 1):
            for c2 in range(c1 + 1, k + 1):
                x1 = color_variable(v, c1, k)
                x2 = color_variable(v, c2, k)

                cnf.append([-x1, -x2])
    # Adjacent vertices cannot have the same color
    for u, v in edges:
        for c in range(1, k + 1):
            x_u = color_variable(u, c, k)
            x_v = color_variable(v, c, k)
            cnf.append([-x_u, -x_v])
    return cnf

In [10]:
#test graph

graph = """4 4
1 2 3
2 3
3 4
"""

with open("graph.txt", "w") as f:
    f.write(graph)

In [11]:
print(open("graph.txt").read())

4 4
1 2 3
2 3
3 4



In [12]:
print(read_graph("graph.txt"))

(4, [(1, 2), (1, 3), (2, 3), (3, 4)])


## Problem 1b

In [13]:
#Call the solver
#use PySAT to call a SAT solver to decide whether the formula is satisfiable.
def can_color(satgraph, k):

    cnf = graph_coloring_cnf(satgraph, k)

    with Solver(name="glucose3", bootstrap_with=cnf.clauses) as solver:

        return solver.solve()

In [14]:
#Test the CNF
cnf2 = graph_coloring_cnf("graph.txt", 2)
print("k=2:", cnf2.nv, len(cnf2.clauses), can_color("graph.txt", 2))

cnf3 = graph_coloring_cnf("graph.txt", 3)
print("k=3:", cnf3.nv, len(cnf3.clauses), can_color("graph.txt", 3))

k=2: 8 16 False
k=3: 12 28 True


In [15]:
cnf = graph_coloring_cnf("graph.txt", 3)

print("Number of clauses:", len(cnf.clauses))

Number of clauses: 28


In [16]:
#test

print(can_color("graph.txt", 1))

False


In [17]:
print(can_color("graph.txt", 2))

False


In [18]:
print(can_color("graph.txt", 3))

True


In [19]:
print("k = 1:", can_color("graph.txt", 1))
print("k = 2:", can_color("graph.txt", 2))
print("k = 3:", can_color("graph.txt", 3))

k = 1: False
k = 2: False
k = 3: True


## Problem 1C - Test code 

In [20]:
#Measure time
def test_coloring(satgraph, k):

    start = time.perf_counter()

    result = can_color(satgraph, k)

    end = time.perf_counter()

    print("Number of colors:", k)
    print("Colorable:", result)
    print("Runtime:", end - start, "seconds")

In [21]:
#From our actual graph in the exercise 

test_coloring("graph.txt", 2)
test_coloring("graph.txt", 3)
test_coloring("graph.txt", 7)

Number of colors: 2
Colorable: False
Runtime: 0.0006888000061735511 seconds
Number of colors: 3
Colorable: True
Runtime: 0.0012454999960027635 seconds
Number of colors: 7
Colorable: True
Runtime: 0.0010221999837085605 seconds


In [22]:
#testing on specific graph sizes with same number of colors


for n in [10, 20, 30, 40, 50]:

    satgraph = f"vertices_{n}.txt"

    graph = random_graph(n, 50)

    with open(satgraph, "w") as f:
        f.write(graph)

    # Read the graph back to find the actual number of edges
    vertices, edges = read_graph(satgraph)

    print("Graph:", satgraph)
    print("Vertices:", vertices)
    print("Edges:", len(edges))
    print("Edge probability: 50%")
    print()


Graph: vertices_10.txt
Vertices: 10
Edges: 25
Edge probability: 50%

Graph: vertices_20.txt
Vertices: 20
Edges: 91
Edge probability: 50%

Graph: vertices_30.txt
Vertices: 30
Edges: 208
Edge probability: 50%

Graph: vertices_40.txt
Vertices: 40
Edges: 410
Edge probability: 50%

Graph: vertices_50.txt
Vertices: 50
Edges: 608
Edge probability: 50%



In [23]:
with open("vertices_20.txt", "r") as f:
    print(f.read())

20 91
1 2 4 6 7 9 11 15 18 19 20
2 4 7 8 13 14 15 16 18
3 6 13 14 15 18 19 20
4 5 7 8 10 11 16
5 6 10 11 12 13 14 15 17 19 20
6 7 8 12 15 16 17
7 8 9 11 13 15 16 18 19
8 11 12 13 15 17 18 20
9 10 13 17 19
10 11 13 14 16 17 18 19
11 13 17 18 20
12 15 16 17 19
13 15 20
14 16 18 19 20
15 18
16 17
17
18 19
19 20
20


In [24]:
# Preliminary test using 3 colours

for n in [10, 20, 30, 40, 50]:

    satgraph = f"vertices_{n}.txt"

    start = time.perf_counter()

    result = can_color(satgraph, 3)

    end = time.perf_counter()

    vertices, edges = read_graph(satgraph)

    print(
        "Graph:", satgraph,
        "| Vertices:", vertices,
        "| Edges:", len(edges),
        "| Colours:", 3,
        "| Result:", result,
        "| Runtime:", end - start,
        "seconds"
    )

Graph: vertices_10.txt | Vertices: 10 | Edges: 25 | Colours: 3 | Result: False | Runtime: 0.001185899949632585 seconds
Graph: vertices_20.txt | Vertices: 20 | Edges: 91 | Colours: 3 | Result: False | Runtime: 0.0019816000130958855 seconds
Graph: vertices_30.txt | Vertices: 30 | Edges: 208 | Colours: 3 | Result: False | Runtime: 0.003859699994791299 seconds
Graph: vertices_40.txt | Vertices: 40 | Edges: 410 | Colours: 3 | Result: False | Runtime: 0.00368190003791824 seconds
Graph: vertices_50.txt | Vertices: 50 | Edges: 608 | Colours: 3 | Result: False | Runtime: 0.005404299998190254 seconds


In [25]:
# Preliminary test using 3 colours

for n in [10, 20, 30, 40, 50]:

    satgraph = f"vertices_{n}.txt"

    start = time.perf_counter()

    result = can_color(satgraph, 10)

    end = time.perf_counter()

    vertices, edges = read_graph(satgraph)

    print(
        "Graph:", satgraph,
        "| Vertices:", vertices,
        "| Edges:", len(edges),
        "| Colours:", 10,
        "| Result:", result,
        "| Runtime:", end - start,
        "seconds"
    )

Graph: vertices_10.txt | Vertices: 10 | Edges: 25 | Colours: 10 | Result: True | Runtime: 0.002812999999150634 seconds
Graph: vertices_20.txt | Vertices: 20 | Edges: 91 | Colours: 10 | Result: True | Runtime: 0.006639899977017194 seconds
Graph: vertices_30.txt | Vertices: 30 | Edges: 208 | Colours: 10 | Result: True | Runtime: 0.00870150001719594 seconds
Graph: vertices_40.txt | Vertices: 40 | Edges: 410 | Colours: 10 | Result: True | Runtime: 0.013087900006212294 seconds
Graph: vertices_50.txt | Vertices: 50 | Edges: 608 | Colours: 10 | Result: True | Runtime: 0.026466600014828146 seconds


In [26]:
# Test Experiment 1 - Different graph sizes using 4 colors

for n in [10, 20, 30, 40, 50]:

    satgraph = f"vertices_{n}.txt"

    start = time.perf_counter()

    result = can_color(satgraph, 4)

    end = time.perf_counter()

    vertices, edges = read_graph(satgraph)

    print(
        "Graph:", satgraph,
        "| Vertices:", vertices,
        "| Edges:", len(edges),
        "| Colours:", 4,
        "| Result:", result,
        "| Runtime:", end - start,
        "seconds"
    )

Graph: vertices_10.txt | Vertices: 10 | Edges: 25 | Colours: 4 | Result: True | Runtime: 0.0017148999613709748 seconds
Graph: vertices_20.txt | Vertices: 20 | Edges: 91 | Colours: 4 | Result: False | Runtime: 0.003626100020483136 seconds
Graph: vertices_30.txt | Vertices: 30 | Edges: 208 | Colours: 4 | Result: False | Runtime: 0.0038030999712646008 seconds
Graph: vertices_40.txt | Vertices: 40 | Edges: 410 | Colours: 4 | Result: False | Runtime: 0.00526849995367229 seconds
Graph: vertices_50.txt | Vertices: 50 | Edges: 608 | Colours: 4 | Result: False | Runtime: 0.008370099996682256 seconds


In [27]:
# Experiment 2: Different edge densities
#Each possible edge has a specified probability of being included in the graph

for probability in [10, 30, 50, 70, 90]:

    filename = f"density_{probability}.txt"

    graph = random_graph(20, probability)

    with open(filename, "w") as f:
        f.write(graph)

    # Find the actual number of edges
    vertices, edges = read_graph(filename)

    print(
        "Graph:", filename,
        "| Vertices:", vertices,
        "| Actual edges:", len(edges),
        "| Edge probability:", probability, "%"
    )

Graph: density_10.txt | Vertices: 20 | Actual edges: 28 | Edge probability: 10 %
Graph: density_30.txt | Vertices: 20 | Actual edges: 55 | Edge probability: 30 %
Graph: density_50.txt | Vertices: 20 | Actual edges: 112 | Edge probability: 50 %
Graph: density_70.txt | Vertices: 20 | Actual edges: 129 | Edge probability: 70 %
Graph: density_90.txt | Vertices: 20 | Actual edges: 172 | Edge probability: 90 %


In [28]:
# Edge-density Experiment 

for probability in [10, 30, 50, 70, 90]:

    filename = f"density_{probability}.txt"

    start = time.perf_counter()

    result = can_color(filename, 4)

    end = time.perf_counter()

    vertices, edges = read_graph(filename)

    print(
        "Graph:", filename,
        "| Vertices:", vertices,
        "| Actual edges:", len(edges),
        "| Edge probability:", probability, "%",
        "| Colours:", 4,
        "| Result:", result,
        "| Runtime:", end - start,
        "seconds"
    )

Graph: density_10.txt | Vertices: 20 | Actual edges: 28 | Edge probability: 10 % | Colours: 4 | Result: True | Runtime: 0.0015940999728627503 seconds
Graph: density_30.txt | Vertices: 20 | Actual edges: 55 | Edge probability: 30 % | Colours: 4 | Result: True | Runtime: 0.0020528999739326537 seconds
Graph: density_50.txt | Vertices: 20 | Actual edges: 112 | Edge probability: 50 % | Colours: 4 | Result: False | Runtime: 0.0021972000249661505 seconds
Graph: density_70.txt | Vertices: 20 | Actual edges: 129 | Edge probability: 70 % | Colours: 4 | Result: False | Runtime: 0.00242390000494197 seconds
Graph: density_90.txt | Vertices: 20 | Actual edges: 172 | Edge probability: 90 % | Colours: 4 | Result: False | Runtime: 0.0025486000231467187 seconds


In [29]:
# Experiment 3: Create one fixed graph

colour_graph = random_graph(20, 30)

with open("colour_test.txt", "w") as f:
    f.write(colour_graph)

vertices, edges = read_graph("colour_test.txt")

print("Graph: colour_test.txt")
print("Vertices:", vertices)
print("Actual edges:", len(edges))
print("Edge probability: 30%")

Graph: colour_test.txt
Vertices: 20
Actual edges: 53
Edge probability: 30%


In [30]:
# Test different numbers of colours on the SAME graph

for k in [2, 3, 4, 5]:

    start = time.perf_counter()

    result = can_color("colour_test.txt", k)

    end = time.perf_counter()

    print(
        "Colours:", k,
        "| Result:", result,
        "| Runtime:", end - start,
        "seconds"
    )

Colours: 2 | Result: False | Runtime: 0.0030266999965533614 seconds
Colours: 3 | Result: False | Runtime: 0.0014497999800369143 seconds
Colours: 4 | Result: True | Runtime: 0.00146240001777187 seconds
Colours: 5 | Result: True | Runtime: 0.001833300048019737 seconds


## Problem 2

In [31]:
#variables

def removal_variable(vertex, color):
    """
    color:
        1 = Red
        2 = Green
        3 = Removed
    """

    return 3 * (vertex - 1) + color

In [32]:
print(removal_variable(1, 1))
print(removal_variable(1, 2))
print(removal_variable(1, 3))

print(removal_variable(2, 1))
print(removal_variable(2, 2))
print(removal_variable(2, 3))

1
2
3
4
5
6


## Problem 2a

In [33]:
#SAT reduction

def removal_cnf(satgraph, k):

    n, edges = read_graph(satgraph)

    cnf = CNF()

    # Each vertex gets exactly one of:
    # R = red
    # G = green
    # S = removed

    for v in range(1, n + 1):

        R = removal_variable(v, 1)
        G = removal_variable(v, 2)
        S = removal_variable(v, 3)

        # At least one color
        cnf.append([R, G, S])

        # At most one color
        cnf.append([-R, -G])
        cnf.append([-R, -S])
        cnf.append([-G, -S])

    # Adjacent vertices cannot have the same
    # red or green color.

    for u, v in edges:

        R_u = removal_variable(u, 1)
        G_u = removal_variable(u, 2)

        R_v = removal_variable(v, 1)
        G_v = removal_variable(v, 2)

        # u and v cannot both be red
        cnf.append([-R_u, -R_v])

        # u and v cannot both be green
        cnf.append([-G_u, -G_v])

    # At most k vertices can be removed

    star_variables = []

    for v in range(1, n + 1):
        star_variables.append(removal_variable(v, 3))
        
   #At most k vertices maybe removed
    
    cardinality = CardEnc.atmost(
        lits=star_variables,
        bound=k,
        top_id=3 * n
    )

    cnf.extend(cardinality.clauses)

    return cnf

In [34]:
cnf2 = removal_cnf("graph.txt", 0)

print("Number of clauses:", len(cnf2.clauses))

Number of clauses: 28


In [35]:
#Decision solver problem

def can_remove_k(satgraph, k):

    cnf = removal_cnf(satgraph, k)

    with Solver(
        name="glucose3",
        bootstrap_with=cnf.clauses
    ) as solver:

        return solver.solve()

In [36]:
print("k = 0:", can_remove_k("graph.txt", 0))
print("k = 1:", can_remove_k("graph.txt", 1))
print("k = 2:", can_remove_k("graph.txt", 2))

k = 0: False
k = 1: True
k = 2: True


In [37]:
def get_removed_vertices(satgraph, k):

    cnf = removal_cnf(satgraph, k)

    with Solver(
        name="glucose3",
        bootstrap_with=cnf.clauses
    ) as solver:

        if not solver.solve():
            return None

        model = set(solver.get_model())

    n, edges = read_graph(satgraph)

    W = []

    for v in range(1, n + 1):

        S = removal_variable(v, 3)

        if S in model:
            W.append(v)

    return W

In [38]:
print("k = 0:", get_removed_vertices("graph.txt", 0))
print("k = 1:", get_removed_vertices("graph.txt", 1))
print("k = 2:", get_removed_vertices("graph.txt", 2))

k = 0: None
k = 1: [2]
k = 2: [3]


In [39]:
for k in range(5):
    result = can_remove_k("graph.txt", k)
    W = get_removed_vertices("graph.txt", k)

    print("k =", k, "| Result:", result, "| W =", W)

k = 0 | Result: False | W = None
k = 1 | Result: True | W = [2]
k = 2 | Result: True | W = [3]
k = 3 | Result: True | W = [1]
k = 4 | Result: True | W = [1]


In [40]:
path_graph = """4 3
1 2
2 3
3 4
"""

with open("path.txt", "w") as f:
    f.write(path_graph)

In [41]:
for k in range(3):
    print(
        "k =", k,
        "| Result:", can_remove_k("path.txt", k),
        "| W =", get_removed_vertices("path.txt", k)
    )

k = 0 | Result: True | W = []
k = 1 | Result: True | W = []
k = 2 | Result: True | W = []


In [42]:
two_triangles = """6 6
1 2 3
2 3
3 1
4 5 6
5 6
6 4
"""

with open("two_triangles.txt", "w") as f:
    f.write(two_triangles)

In [43]:
for k in range(5):
    print(
        "k =", k,
        "| Possible:", can_remove_k("two_triangles.txt", k),
        "| W =", get_removed_vertices("two_triangles.txt", k)
    )

k = 0 | Possible: False | W = None
k = 1 | Possible: False | W = None
k = 2 | Possible: True | W = [2, 5]
k = 3 | Possible: True | W = [2, 4]
k = 4 | Possible: True | W = [3, 4]


In [44]:
def test_decision(satgraph, k):

    start = time.perf_counter()

    result = can_remove_k(satgraph, k)

    end = time.perf_counter()

    print("Graph:", satgraph)
    print("k:", k)
    print("Possible:", result)
    print("Runtime:", end - start, "seconds")

In [45]:
test_decision("graph.txt", 0)
test_decision("path.txt", 0)
test_decision("two_triangles.txt", 1)

Graph: graph.txt
k: 0
Possible: False
Runtime: 0.0009815000230446458 seconds
Graph: path.txt
k: 0
Possible: True
Runtime: 0.0013291999930515885 seconds
Graph: two_triangles.txt
k: 1
Possible: False
Runtime: 0.000838099978864193 seconds


## Problem 2b

In [46]:
#Problem 2b

def minimum_removed_vertices(satgraph):

    n, edges = read_graph(satgraph)

    number_of_calls = 0

    for k in range(n + 1):

        number_of_calls += 1

        if can_remove_k(satgraph, k):

            W = get_removed_vertices(satgraph, k)

            return k, W, number_of_calls

In [47]:
result = minimum_removed_vertices("graph.txt")

print("Minimum k:", result[0])
print("Removed vertices W:", result[1])
print("Number of Decision SAT calls:", result[2])


Minimum k: 1
Removed vertices W: [2]
Number of Decision SAT calls: 2


In [48]:
result = minimum_removed_vertices("path.txt")

print("Minimum k:", result[0])
print("Removed vertices W:", result[1])
print("Number of Decision SAT calls:", result[2])

Minimum k: 0
Removed vertices W: []
Number of Decision SAT calls: 1


In [49]:
result = minimum_removed_vertices("two_triangles.txt")

print("Minimum k:", result[0])
print("Removed vertices W:", result[1])
print("Number of Decision SAT calls:", result[2])

Minimum k: 2
Removed vertices W: [2, 5]
Number of Decision SAT calls: 3


## Problem 2C

In [50]:
def test_minimization(satgraph):

    start = time.perf_counter()

    k, W, calls = minimum_removed_vertices(satgraph)

    end = time.perf_counter()

    vertices, edges = read_graph(satgraph)

    print("Graph:", satgraph)
    print("Vertices:", vertices)
    print("Edges:", len(edges))
    print("Minimum number of removed vertices:", k)
    print("Removed vertices W:", W)
    print("Number of Decision SAT calls:", calls)
    print("Runtime:", end - start, "seconds")
    

In [51]:
test_minimization("graph.txt")

Graph: graph.txt
Vertices: 4
Edges: 4
Minimum number of removed vertices: 1
Removed vertices W: [2]
Number of Decision SAT calls: 2
Runtime: 0.004816900007426739 seconds


In [52]:
test_minimization("path.txt")

Graph: path.txt
Vertices: 4
Edges: 3
Minimum number of removed vertices: 0
Removed vertices W: []
Number of Decision SAT calls: 1
Runtime: 0.0029776999726891518 seconds


In [53]:
test_minimization("two_triangles.txt")

Graph: two_triangles.txt
Vertices: 6
Edges: 8
Minimum number of removed vertices: 2
Removed vertices W: [2, 5]
Number of Decision SAT calls: 3
Runtime: 0.004339999984949827 seconds


In [54]:
larger_graph = """8 13
1 2 3 4
2 3 5
3 4 6
4 5 7
5 6 8
6 7
7 8
"""

with open("larger.txt", "w") as f:
    f.write(larger_graph)

In [55]:
print(read_graph("larger.txt"))

(8, [(1, 2), (1, 3), (1, 4), (2, 3), (2, 5), (3, 4), (3, 6), (4, 5), (4, 7), (5, 6), (5, 8), (6, 7), (7, 8)])


In [56]:
test_minimization("larger.txt")

Graph: larger.txt
Vertices: 8
Edges: 13
Minimum number of removed vertices: 1
Removed vertices W: [3]
Number of Decision SAT calls: 2
Runtime: 0.00373070000205189 seconds


In [57]:
# Create larger graphs

for n in [10, 20, 30, 40]:

    filename = f"large_{n}.txt"

    graph = random_graph(n, 30)

    with open(filename, "w") as f:
        f.write(graph)

    vertices, edges = read_graph(filename)

    print(
        "Graph:", filename,
        "| Vertices:", vertices,
        "| Actual edges:", len(edges),
        "| Edge probability: 30%"
    )

Graph: large_10.txt | Vertices: 10 | Actual edges: 10 | Edge probability: 30%
Graph: large_20.txt | Vertices: 20 | Actual edges: 54 | Edge probability: 30%
Graph: large_30.txt | Vertices: 30 | Actual edges: 144 | Edge probability: 30%
Graph: large_40.txt | Vertices: 40 | Actual edges: 232 | Edge probability: 30%


In [58]:
# Test the larger graphs

for n in [10, 20, 30, 40]:

    filename = f"large_{n}.txt"

    test_minimization(filename)

Graph: large_10.txt
Vertices: 10
Edges: 10
Minimum number of removed vertices: 0
Removed vertices W: []
Number of Decision SAT calls: 1
Runtime: 0.0030873999930918217 seconds
Graph: large_20.txt
Vertices: 20
Edges: 54
Minimum number of removed vertices: 6
Removed vertices W: [7, 9, 10, 13, 15, 17]
Number of Decision SAT calls: 7
Runtime: 0.015434300003107637 seconds
Graph: large_30.txt
Vertices: 30
Edges: 144
Minimum number of removed vertices: 13
Removed vertices W: [2, 3, 4, 5, 6, 11, 12, 15, 16, 24, 25, 27, 28]
Number of Decision SAT calls: 14
Runtime: 0.12353039998561144 seconds
Graph: large_40.txt
Vertices: 40
Edges: 232
Minimum number of removed vertices: 21
Removed vertices W: [1, 3, 5, 6, 10, 15, 17, 20, 21, 23, 25, 26, 27, 28, 31, 32, 33, 35, 37, 38, 40]
Number of Decision SAT calls: 22
Runtime: 2.2840591999702156 seconds


In [5]:
def is_one_colorable(vertices, edges):
    # A graph is 1-colorable if it has no edges
    return len(edges) == 0


# One vertex: A
vertices = ["A"]
edges = []

print(is_one_colorable(vertices, edges))

True


In [6]:
vertices = ["A", "B"]
edges = [("A", "B")]

print(len(edges) == 0)

False


In [7]:
vertices = ["A", "B"]
edges = []

print(len(edges) == 0)

True
